# Canonical Quantitative Manifest and Reconciliation

Builds the versioned 71-knee planning manifest, explicit excluded rows, five subject-grouped folds, slice groups and leakage checks. Rows remain pending_recertification until versioned outputs and returned HPC evidence are approved. File existence alone never marks a row ready.

In [1]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
import nibabel as nib
import pydicom
from sklearn.model_selection import StratifiedGroupKFold

ROOT = Path.cwd().resolve()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "data").exists(), "Could not locate project data directory"
REPORT_DIR = ROOT / "reports" / "manifests"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
DATA_CONTRACT = json.loads((ROOT / "configs" / "data_contract_v1.json").read_text(encoding="utf-8"))
CERTIFICATION_APPROVED = False
SEED = 42

In [2]:
def rel(path):
    if path is None:
        return ""
    path = Path(path)
    try:
        return path.relative_to(ROOT).as_posix()
    except ValueError:
        return path.as_posix()


def sample_paths(sample_id, cohort, side):
    if cohort == "healthy":
        case = sample_id.rsplit("_", 1)[0]
    else:
        case = sample_id.split("_Part", 1)[0]
    side_lower = side.lower()
    return {
        "predrr_path": f"data/interim/predrr_lps_256_v1/{cohort}/{sample_id}.nii.gz",
        "ap_drr_path": f"data/interim/DRRs_diffdrr_lps_256_v1/{cohort}/{case}/{side_lower}/ap.npy",
        "lat_drr_path": f"data/interim/DRRs_diffdrr_lps_256_v1/{cohort}/{case}/{side_lower}/lat.npy",
        "target_path": f"data/interim/gt_per_bone_lps_256_v1/{cohort}/{sample_id}",
        "fracture_roi_path": (
            f"data/interim/fracture_roi_lps_256_v1/{sample_id}_fracture_roi.nii.gz"
            if cohort == "fractured" else ""
        ),
    }


def dicom_spacing_and_group(folder):
    files = sorted(Path(folder).glob("*.dcm"))
    if not files:
        return None, "unknown"
    headers = [pydicom.dcmread(str(p), stop_before_pixels=True, force=True) for p in files]
    first = headers[0]
    pixel = [float(v) for v in getattr(first, "PixelSpacing", [np.nan, np.nan])]
    positions = []
    for ds in headers:
        ipp = getattr(ds, "ImagePositionPatient", None)
        if ipp is not None:
            positions.append(np.asarray([float(v) for v in ipp]))
    z_spacing = float(getattr(first, "SpacingBetweenSlices", getattr(first, "SliceThickness", np.nan)))
    if len(positions) >= 2:
        direction = np.asarray([float(v) for v in getattr(first, "ImageOrientationPatient", [1,0,0,0,1,0])])
        normal = np.cross(direction[:3], direction[3:])
        projected = np.sort(np.asarray(positions) @ normal)
        diffs = np.diff(projected)
        diffs = np.abs(diffs[diffs != 0])
        if len(diffs):
            z_spacing = float(np.median(diffs))
    if np.isfinite(z_spacing):
        group = "thin_0.7mm" if z_spacing < 1.5 else "thick_3.0mm"
    else:
        group = "unknown"
    return [pixel[1], pixel[0], z_spacing], group


def outputs_present(row):
    paths = [row[k] for k in ("predrr_path", "ap_drr_path", "lat_drr_path")]
    target_dir = ROOT / row["target_path"]
    bone_files = [target_dir / f"{row['sample_id']}_{bone}.nii.gz" for bone in DATA_CONTRACT["bone_channels"]]
    return all((ROOT / p).is_file() for p in paths) and all(p.is_file() for p in bone_files)

In [3]:
rows = []
healthy_files = sorted((ROOT / "data" / "raw" / "healthy").rglob("VSD_*.nii.gz"))
for source in healthy_files:
    sample_id = source.name.removesuffix(".nii.gz")
    subject_id, side = sample_id.rsplit("_", 1)
    image = nib.load(str(source))
    native_spacing = [float(v) for v in image.header.get_zooms()[:3]]
    row = {
        "sample_id": sample_id,
        "subject_id": subject_id,
        "dataset": "VSD",
        "side": side,
        "fracture_status": "healthy",
        "status": "pending_recertification",
        "exclusion_reason": "",
        "source_class": "knee_crop_nifti",
        "source_path": rel(source),
        "native_spacing_xyz_mm": json.dumps(native_spacing),
        "final_spacing_xyz_mm": json.dumps(DATA_CONTRACT["final_spacing_mm"]),
        "orientation": "LPS",
        "laterality_verified": False,
        "target_version": DATA_CONTRACT["target_version"],
        "drr_version": DATA_CONTRACT["drr_version"],
        "roi_version": "",
        "slice_group": "not_applicable",
        "test_fold": pd.NA,
        "augmentation_parent": sample_id,
        "fracture_roi_status": "not_applicable",
    }
    row.update(sample_paths(sample_id, "healthy", side))
    rows.append(row)

for part_name, side in (("PartLeft", "Left"), ("PartRight", "Right")):
    part_dir = ROOT / "data" / "raw" / "fractured" / part_name
    for case_dir in sorted(p for p in part_dir.iterdir() if p.is_dir()):
        sample_id = f"{case_dir.name}_{part_name}"
        spacing, slice_group = dicom_spacing_and_group(case_dir)
        excluded = case_dir.name in {"Case4", "Case8", "Case10"}
        reason = {
            "Case4": "scout_localizer",
            "Case8": "fracture_fixation_implant",
            "Case10": "scout_localizer",
        }.get(case_dir.name, "")
        row = {
            "sample_id": sample_id,
            "subject_id": case_dir.name,
            "dataset": "Ruikar",
            "side": side,
            "fracture_status": "fractured",
            "status": "excluded" if excluded else "pending_recertification",
            "exclusion_reason": reason,
            "source_class": "dicom_series",
            "source_path": rel(case_dir),
            "native_spacing_xyz_mm": json.dumps(spacing),
            "final_spacing_xyz_mm": json.dumps(DATA_CONTRACT["final_spacing_mm"]),
            "orientation": "LPS",
            "laterality_verified": False,
            "target_version": DATA_CONTRACT["target_version"],
            "drr_version": DATA_CONTRACT["drr_version"],
            "roi_version": DATA_CONTRACT["roi_version"],
            "slice_group": slice_group,
            "test_fold": pd.NA,
            "augmentation_parent": sample_id,
            "fracture_roi_status": "not_applicable" if excluded else "needs_user_review",
        }
        row.update(sample_paths(sample_id, "fractured", side))
        rows.append(row)

for sample_id, subject, side, reason in [
    ("VSD_z057_Left", "VSD_z057", "Left", "metal_artifact"),
    ("VSD_z057_Right", "VSD_z057", "Right", "metal_artifact"),
    ("VSD_z050_Left", "VSD_z050", "Left", "tkr_metal_artifact_lps_centroid"),
    ("VSD_z063_Left", "VSD_z063", "Left", "tkr_metal_artifact_lps_centroid"),
]:
    rows.append({
        "sample_id": sample_id, "subject_id": subject, "dataset": "VSD", "side": side,
        "fracture_status": "healthy", "status": "excluded", "exclusion_reason": reason,
        "source_class": "excluded_source_record", "source_path": "",
        "native_spacing_xyz_mm": "", "final_spacing_xyz_mm": json.dumps(DATA_CONTRACT["final_spacing_mm"]),
        "orientation": "LPS", "laterality_verified": False,
        "predrr_path": "", "ap_drr_path": "", "lat_drr_path": "", "target_path": "",
        "fracture_roi_path": "", "target_version": DATA_CONTRACT["target_version"],
        "drr_version": DATA_CONTRACT["drr_version"], "roi_version": "",
        "slice_group": "not_applicable", "test_fold": pd.NA,
        "augmentation_parent": sample_id, "fracture_roi_status": "not_applicable",
    })

manifest = pd.DataFrame(rows)
manifest["laterality_audit_status"] = "not_applicable"
laterality_audit_path = REPORT_DIR / "vsd_cohort_laterality_multibone_v2.csv"
if laterality_audit_path.exists():
    laterality_audit = pd.read_csv(laterality_audit_path).set_index("sample_id")
    mapped_status = manifest["sample_id"].map(laterality_audit["status"])
    manifest.loc[mapped_status.notna(), "laterality_audit_status"] = mapped_status[mapped_status.notna()]
    manifest.loc[mapped_status.eq("PASS"), "laterality_verified"] = True
manifest.loc[manifest.sample_id.isin(["VSD_z050_Right", "VSD_z063_Right"]), "laterality_verified"] = True
manifest.loc[manifest.sample_id.isin(["VSD_z050_Left", "VSD_z063_Left"]), "laterality_verified"] = True
included = manifest[manifest["status"] != "excluded"].copy()
assert len(included) == 71, f"expected 71 included planning rows, found {len(included)}"
assert (included["fracture_status"] == "healthy").sum() == 58
assert (included["fracture_status"] == "fractured").sum() == 13
assert manifest["sample_id"].is_unique

In [4]:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
fold_by_index = {}
x = np.zeros((len(included), 1))
y = included["fracture_status"].to_numpy()
groups = included["subject_id"].to_numpy()
for fold, (_, test_idx) in enumerate(sgkf.split(x, y, groups)):
    for idx in test_idx:
        fold_by_index[included.index[idx]] = fold
manifest.loc[list(fold_by_index), "test_fold"] = [fold_by_index[i] for i in fold_by_index]
manifest["test_fold"] = manifest["test_fold"].astype("Int64")

subject_fold_counts = manifest.loc[manifest.status != "excluded"].groupby("subject_id")["test_fold"].nunique()
assert subject_fold_counts.max() == 1
assert set(manifest.loc[manifest.status != "excluded", "test_fold"].dropna().astype(int)) == set(range(5))
assert not (manifest["test_fold"] == 5).any()
assert (manifest.loc[manifest.status != "excluded", "augmentation_parent"] == manifest.loc[manifest.status != "excluded", "sample_id"]).all()

if CERTIFICATION_APPROVED:
    present = manifest.apply(lambda row: row.status == "excluded" or outputs_present(row), axis=1)
    assert present.all(), "certification requested but one or more version-matched artifact sets are missing"
    manifest.loc[manifest.status != "excluded", "status"] = "ready"

columns = [
    "sample_id", "subject_id", "dataset", "side", "fracture_status", "status", "exclusion_reason",
    "source_class", "source_path", "native_spacing_xyz_mm", "final_spacing_xyz_mm", "orientation",
    "laterality_verified", "predrr_path", "ap_drr_path", "lat_drr_path", "target_path",
    "fracture_roi_path", "target_version", "drr_version", "roi_version", "slice_group",
    "test_fold", "augmentation_parent", "fracture_roi_status", "laterality_audit_status",
]
manifest = manifest[columns].sort_values(["status", "dataset", "subject_id", "side"]).reset_index(drop=True)
csv_path = REPORT_DIR / "quantitative_manifest_v1.csv"
manifest.to_csv(csv_path, index=False)
sha = hashlib.sha256(csv_path.read_bytes()).hexdigest()
(REPORT_DIR / "quantitative_manifest_v1.sha256").write_text(
    f"{sha}  quantitative_manifest_v1.csv\n", encoding="utf-8"
)

flow = manifest.groupby(["dataset", "fracture_status", "status", "exclusion_reason"], dropna=False).size().rename("n").reset_index()
flow.to_csv(REPORT_DIR / "cohort_flow_v1.csv", index=False)
leakage = {
    "subject_multiple_test_folds": int((subject_fold_counts > 1).sum()),
    "fold5_rows": int((manifest["test_fold"] == 5).sum()),
    "augmentation_parent_mismatch": int(
        (manifest.loc[manifest.status != "excluded", "augmentation_parent"] !=
         manifest.loc[manifest.status != "excluded", "sample_id"]).sum()
    ),
}
metadata = {
    "manifest_version": "quantitative_manifest_v1",
    "sha256": sha,
    "seed": SEED,
    "n_splits": 5,
    "validation_rule": "(test_fold + 1) mod 5",
    "included_planning_rows": int((manifest.status != "excluded").sum()),
    "ready_rows": int((manifest.status == "ready").sum()),
    "pending_recertification_rows": int((manifest.status == "pending_recertification").sum()),
    "excluded_rows": int((manifest.status == "excluded").sum()),
    "certification_approved": CERTIFICATION_APPROVED,
    "leakage": leakage,
    "blocking_notes": [
        "VSD010 merge geometry passes; the four-bone multibone v2 audit supersedes absolute-X inference and passes 58/58 retained VSD knees.",
        "z050/z063 Right survivors and Left TKR exclusions are user-approved.",
        "Focused CT-STL-target review passes z023 Left/Right and z036 Left/Right; no mapping override is required.",
        "Versioned predrr, four-bone targets and AP/LAT DRRs require returned HPC evidence.",
        "Manual Ruikar fracture ROI statuses require user review.",
    ],
}
(REPORT_DIR / "quantitative_manifest_v1.metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
(REPORT_DIR / "leakage_report_v1.json").write_text(json.dumps(leakage, indent=2), encoding="utf-8")
print(json.dumps(metadata, indent=2))
assert all(v == 0 for v in leakage.values())

{
  "manifest_version": "quantitative_manifest_v1",
  "sha256": "c517ccfe09f5231e39825848a9c0aa2b2df299b69255d57c29100411bcc8ef0e",
  "seed": 42,
  "n_splits": 5,
  "validation_rule": "(test_fold + 1) mod 5",
  "included_planning_rows": 71,
  "ready_rows": 0,
  "pending_recertification_rows": 71,
  "excluded_rows": 7,
  "certification_approved": false,
  "leakage": {
    "subject_multiple_test_folds": 0,
    "fold5_rows": 0,
    "augmentation_parent_mismatch": 0
  },
  "blocking_notes": [
    "VSD010 merge geometry passes; the four-bone multibone v2 audit supersedes absolute-X inference and passes 58/58 retained VSD knees.",
    "z050/z063 Right survivors and Left TKR exclusions are user-approved.",
    "Focused CT-STL-target review passes z023 Left/Right and z036 Left/Right; no mapping override is required.",
    "Versioned predrr, four-bone targets and AP/LAT DRRs require returned HPC evidence.",
    "Manual Ruikar fracture ROI statuses require user review."
  ]
}


## Success and current verdict

The planning cohort and grouped folds pass when the notebook reports 71 included rows and zero leakage. Stage 1 certification remains BLOCKED while any included row is pending_recertification, either TKR side is unapproved, VSD010 merge evidence is pending, or an HPC preprocessing bundle is unreviewed.